In [ ]:
def get_pLDDT(pdb_p):
    infile = open(pdb_p,'r')
    pLDDT = []
    for line in infile:
          if line[0:6].rstrip() == 'ATOM' and line[13:15].rstrip() == 'CA':
            pLDDT.append(float(line[60:65].rstrip()))
    infile.close()
    return np.mean(pLDDT)

In [ ]:
from CLEAN.utils import *
dirs = "../local_snapshot/tttianhao-CLEAN-0cf2cac/"
id_ec, ec_id =  get_ec_id_dict(dirs+'data/split100' + '.csv')

In [ ]:
def custom_sort(version):
    parts = version.split('.')
    converted_parts = []
    for part in parts:
        try:
            converted_parts.append(int(part))
        except ValueError:
            # If part is not a valid integer, assign a large value to it
            converted_parts.append(float('inf'))
    return tuple(converted_parts)


keys = sorted(ec_id.keys(), key=custom_sort)

In [ ]:
key_class = {}
i = 0
for key in keys:
    i+=1
    key_class[key] = i
    
file = '../local_snapshot/data/fine_tune_data.csv'
fune_tune_file = open(file, 'w')
head = 'sequence'+',' + 'label' + '\n'
fune_tune_file.write(head)
for key in ec_id.keys():
    for ids in ec_id[key]:
        #print(ids)
        sequence = file_table[file_table['Entry']==ids]['Sequence'].iloc[0] 
        label = key_class[key]
        fune_tune_file.write(sequence+','+ str(label)+'\n')
        

In [ ]:
file3 = '../local_snapshot/data/all/all_cluster.tsv'
file4 = '../local_snapshot/data/all/afall_ID50_cluster.tsv'
file5 = '../local_snapshot/data/all/select_cluster.tsv'

In [ ]:
rep_seq = {}
rep_fold = {}
seqfile = open(file3,'r')
for line in seqfile:
    key = line.split('\t')[0]
    if key in rep_seq.keys():
        rep_seq[key].append(line.split('\t')[1].split('\n')[0])
    else:
        rep_seq[key] = []
        rep_seq[key].append(line.split('\t')[1].split('\n')[0])
        
foldfile = open(file5,'r')
for line in foldfile:
    key = line.split('\t')[0].split('.')[0]
    if key in rep_fold.keys():
        rep_fold[key].append(line.split('\t')[1].split('.')[0])
    else:
        rep_fold[key] = []
        rep_fold[key].append(line.split('\t')[1].split('.')[0])

In [ ]:
import os
import numpy as np
af_path = '../local_snapshot/data/afall/'
af_list = os.listdir(af_path)
rep_structure = []
rep_pLDDT = []
keys = []
for key in rep_seq.keys():
    pLDDTs = []
    items = []
    for item in rep_seq[key]:
        item_name = item + '.pdb'
        if item_name in af_list:
            pdb_p = af_path+ item + '.pdb'
            pLDDTs.append(get_pLDDT(pdb_p))
            items.append(item)
    if len(pLDDTs)>0:
        max_plddt = max(pLDDTs)
        rep_pLDDT.append(max_plddt)
        rep_structure.append(items[pLDDTs.index(max_plddt)])
        keys.append(key)

In [ ]:
threshold_value = 90 
positions = [index for index, value in enumerate(rep_pLDDT) if value > threshold_value]

sel_struc=[rep_structure[index ]for index in positions]
sel_keys = [keys[index ]for index in positions]


In [ ]:
fun_fold = {}
sort_items = []
sel_seq = {}
for key in rep_fold.keys():
    fun_fold[key] = []
    
    for item in rep_fold[key]:
        if item in sel_struc:
            k = sel_keys[sel_struc.index(item)]
            fun_fold[key].extend(rep_seq[k])
            sel_seq[k] = rep_seq[k]
        
            sort_items.extend(rep_seq[k])
        #print(key,item,len(sort_items))
        if item not in sel_struc and len(rep_fold[key])==1:
            fun_fold[key].append(key)
            sort_items.append(key)

In [ ]:
sel_seq = {}
for item in sel_struc:
    k = sel_keys[sel_struc.index(item)]
    sel_seq[k] = rep_seq[k]
    
members = []
keys = []
for key in sel_seq.keys():
    keys.append(key)
    members.append(len(sel_seq[key]))
    
sing_seq = [keys[index] for index, value in enumerate(members) if value == 1]

In [ ]:
members = []
keys = []
for key in rep_fold.keys():
    keys.append(key)
    members.append(len(rep_fold[key]))
sing_fold = [keys[index] for index, value in enumerate(members) if value == 1]

In [ ]:
members = []
keys = []
for key in fun_fold.keys():
    keys.append(key)
    members.append(len(fun_fold[key]))
sing_funfold = [keys[index] for index, value in enumerate(members) if value == 1]

In [ ]:
members = []
keys = []
for key in fun_clu.keys():
    keys.append(key)
    members.append(len(fun_clu[key]))
sing_fold_fun = [keys[index] for index, value in enumerate(members) if value == 1]

In [ ]:
fun_check = {}
fun_key = {}
fold_index = []
fold_items = []
for key in fun_fold.keys():
    fun_key[key] = id_ec[key]
    fun_check[key]= []
    for item in fun_fold[key]:
        fun_check[key].append(id_ec[item])
        fold_index.append(key)
        fold_items.append(item)

fun_clu = {}
for key in ec_id.keys():
    fun_clu[key] = []
    for item in ec_id[key]:
        if item in fold_items:
            fun_clu[key].append(fold_index[fold_items.index(item)])
    if len(fun_clu[key]) == 0:
        fun_clu.pop(key, None)


uni_fold = []
for key in fun_clu.keys():
    list_of_lists=fun_clu[key]
    uni_fold.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))
    
uni_fun = []
for key in fun_check.keys():
    list_of_lists=fun_check[key]
    uni_fun.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))
    


In [ ]:
fun_check_fold = {}
fun_key_fold = {}
fold_index_fold = []
fold_items_fold = []
for key in rep_fold.keys():
    fun_key_fold[key] = id_ec[key]
    fun_check_fold[key]= []
    for item in rep_fold[key]:
        fun_check_fold[key].append(id_ec[item])
        fold_index_fold.append(key)
        fold_items_fold.append(item)

uni_fun_fold = []
for key in fun_check_fold.keys():
    list_of_lists=fun_check_fold[key]
    uni_fun_fold.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))

In [ ]:
keys = []
for key in fun_check_fold.keys():
    keys.append(key)

In [ ]:
len(keys)
uni_fun_fold_keys = []
uni_nofun_fold_keys = []
for i in range(len(keys)):
    if uni_fun_fold[i]:
        uni_fun_fold_keys.append(keys[i])
    else:
        uni_nofun_fold_keys.append(keys[i])

In [ ]:
for item in rep_fold[key]:
    if item in sel_struc:
            k = sel_keys[sel_struc.index(item)]
            fun_fold[key].extend(rep_seq[k])

In [ ]:
def is_consistent(cluster):
    ECs = []
    uni_fun_clu = []
    for item in cluster:
        ECs.append(id_ec[item])
    
    return all(sublist == ECs[0] for sublist in ECs)

In [ ]:
uni_fun_seq = []
for key in rep_seq.keys():
    if is_consistent(rep_seq[key]) == 1:
        uni_fun_seq.append(key)

In [ ]:
uni_fun_seq = []
for key in rep_seq.keys():
    ECs = []
    for item in rep_seq[key]:
        ECs.append(id_ec[item])
    if all(sublist == ECs[0] for sublist in ECs):
        uni_fun_seq.append(key)

In [ ]:
cluster = rep_seq['C5C9S6']
ECs = []
uni_fun_clu = []
for item in cluster:
    ECs.append(id_ec[item])
uni_fun_clu.append(all(sublist == ECs[0] for sublist in ECs))

In [ ]:
for key in uni_nofun_fold_keys:
    items = []
    for item in rep_fold[key]:
        if item in sel_struc:
            k = sel_keys[sel_struc.index(item)]
            if is_consistent(rep_seq[k]):
                items.append(item)
        

In [ ]:
keys = []
for key in fun_clu.keys():
    keys.append(key)
    
uni_fold_clu_key = [keys[i] for i in range(len(uni_fold)) if uni_fold[i]]


uni_fold_clu_key_nsingle = []
for item in uni_fold_clu_key:
    if item not in sing_fold_fun:
        uni_fold_clu_key_nsingle.append(item)




In [ ]:
keys = []
for key in fun_check_fold.keys():
    keys.append(key)
    
uni_fun_clu_key_fold = [keys[i] for i in range(len(uni_fun)) if uni_fun_fold[i]]


uni_fun_clu_key_nsingle_fold = []
for item in uni_fun_clu_key_fold:
    if item not in sing_fold:
        uni_fun_clu_key_nsingle_fold.append(item)

In [ ]:
members = []
keys = []
for key in fun_fold.keys():
    keys.append(key)
    members.append(len(fun_fold[key]))
uni_fun_true_size =[members[i] for i in range(len(uni_fun)) if uni_fun[i] ]
uni_fun_False_size =[members[i] for i in range(len(uni_fun)) if uni_fun[i]==False]

uni_fun_true_key = [keys[i] for i in range(len(uni_fun)) if uni_fun[i] ]
uni_fun_False_key = [keys[i] for i in range(len(uni_fun)) if uni_fun[i]==False]

In [ ]:
uni = []
for key in uni_fun_False_key:
    items = []
    for item in fun_check_fold[key]:
        items.append(item)
    if len(set(np.array(items).flatten())) == 1:
        uni.append(key)
        
        
        

In [ ]:
EC_num = []
for item in uni_fun_true_key:
    EC_num.append(len(set(np.array(fun_check[item]).flatten())))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

def cluster_size_count(your_list):
    counter = Counter(your_list)
    segment_counts = {segment: 0 for segment in segments}
    for element, count in counter.items():
        for segment in segments:
            if segment[0] <= element < segment[1]:
                segment_counts[segment] += count
    return segment_counts

# 定义segments和获取两个数据集的计数
segments = [(1, 2), (2, 4), (4, 10), (10, 50), (50, 100), (100, 300), (300, 600), (600, 900)]
counts_fold = cluster_size_count(uni_fun_true_size)
counts_fli = cluster_size_count(uni_fun_False_size)


# 计算每个段内的百分比
percentages_fold = {segment: (count / (count+counts_fli[segment])) * 100 for segment, count in counts_fold.items()}
percentages_fli = {segment: (count / (count+counts_fold[segment])) * 100 for segment, count in counts_fli.items()}

# 将百分比转换为NumPy数组以便堆积
fold_percentages = np.array(list(percentages_fold.values()))
fli_percentages = np.array(list(percentages_fli.values()))

# 画百分比堆积图
plt.figure(figsize=(8, 6))
plt.bar(range(len(segments)), fold_percentages, color='gray', alpha=0.7, label='EC consistent', tick_label=[f"{segment[0]}-{segment[1]}" for segment in segments])
plt.bar(range(len(segments)), fli_percentages, bottom=fold_percentages, color='salmon', alpha=0.7, label='EC not consistent')

plt.xlabel('Cluster Size')
plt.ylabel('Percentage')
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

segments = [(1,2),(2, 4), (4, 10), (10, 50), (50, 100),(100, 300),(300, 600),(600, 900)]

def cluster_size_count(your_list):

    counter = Counter(your_list)
    #segments = [(50, 100),(100, 200),(200, 350),(350, 500),(500, 700),(700, 900)]

    segment_counts = {segment: 0 for segment in segments}

    for element, count in counter.items():
        for segment in segments:
            if segment[0] <= element < segment[1]:
                segment_counts[segment] += count
    return segment_counts
# 将统计结果转换为两个列表，分别包含段和对应的计数
_,  counts_fold = zip(*cluster_size_count(uni_fun_true_size).items())
_,  counts_fli = zip(*cluster_size_count(uni_fun_False_size).items())
# 画柱状图
plt.figure(figsize=(8, 6))
bars_fold = plt.bar(range(len(segments)), counts_fold,color='gray', alpha=0.5,label='EC consistent', tick_label=[f"{segment[0]}-{segment[1]}" for segment in segments])
bars_fli = plt.bar(range(len(segments)), counts_fli,color='salmon', alpha=0.5,label='EC not consistent' , tick_label=[f"{segment[0]}-{segment[1]}" for segment in segments])

plt.xlabel('Cluster Size')
plt.ylabel('Count')
#plt.title('Segmented Counts')
plt.grid(False)
for bar in bars_fold:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval, round(yval, 2), ha='center', va='top')
for bar in bars_fli:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval, round(yval, 2), ha='center', va='bottom')
plt.legend()
plt.show()

In [ ]:
import h5py
proVec_embfile = '../local_snapshot/data/Enzyme_proteins_embed_dic'
miss_embvalue = '../local_snapshot/data/Miss_proteins_embed_dic'
miss_data = ['P20894', 'A0A0U1LSP0', 'Q03498', 'Q8MU52', 'A0A140CUL9', 'W6JQJ6','B3IVI7', 'Q88QV2']
def load_proVec(lookup):
    if lookup not in miss_data:
        with h5py.File(proVec_embfile , 'r') as hf:
            dataset_name = lookup
            dataset = hf[dataset_name]


            data = dataset[:][0]
    else:
        with h5py.File(miss_embvalue , 'r') as hf:
            dataset_name = lookup
            dataset = hf[dataset_name]


            data = dataset[:][0]

    #torch_data = torch.tensor(data)
    return data #torch_data.unsqueeze(0)


def cos_sim(vector1,vector2):
    # Compute dot product
    dot_product = np.dot(vector1, vector2)

    # Compute magnitudes
    magnitude1 = np.linalg.norm(vector1)
    magnitude2 = np.linalg.norm(vector2)

    # Compute cosine similarity
    similarity = dot_product / (magnitude1 * magnitude2)
    return similarity


from sklearn.metrics.pairwise import cosine_similarity

def foldseek_clusterProvec(key):
    clusterProvec = []
    for item in fun_fold[key]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def repfoldseek_clusterProvec(key):
    clusterProvec = []
    for item in rep_fold[key]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def EC_clusterProvec(EC):
    clusterProvec = []
    for item in ec_id[EC]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def cluster_cos_sim(cluster):

    # Reshape the cluster if needed
    cluster = np.array(cluster).reshape(len(cluster), -1)

    # Compute pairwise cosine similarities
    similarities = cosine_similarity(cluster)
    # 去掉对角线的值
    np.fill_diagonal(similarities, 0)
    mask = np.triu(np.ones(similarities.shape), k=1).astype(bool)
    similarities_half = similarities[mask]
    return similarities_half

def cluster_cos_sim_full(cluster):

    # Reshape the cluster if needed
    cluster = np.array(cluster).reshape(len(cluster), -1)

    # Compute pairwise cosine similarities
    similarities = cosine_similarity(cluster)
    
    return similarities


In [ ]:
import h5py
proVec_embfile = '../local_snapshot/data/Embedding/RepFunEC.h5'
#'../local_snapshot/data/Embedding/sep3rnu5_1000e_s_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/sep3sssrn_1000e_s_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/sep3sr_s_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/sep3as_c_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/split_50_prott_triplet_CLEAN.h5'
def load_proVec(lookup):
    
    with h5py.File(proVec_embfile , 'r') as hf:
            dataset_name = lookup
            dataset = hf[dataset_name]


            data = dataset[:]

    #torch_data = torch.tensor(data)
    return data #torch_data.unsqueeze(0)


def cos_sim(vector1,vector2):
    # Compute dot product
    dot_product = np.dot(vector1, vector2)

    # Compute magnitudes
    magnitude1 = np.linalg.norm(vector1)
    magnitude2 = np.linalg.norm(vector2)

    # Compute cosine similarity
    similarity = dot_product / (magnitude1 * magnitude2)
    return similarity


from sklearn.metrics.pairwise import cosine_similarity

def foldseek_clusterProvec(key):
    clusterProvec = []
    for item in fun_fold[key]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def repfoldseek_clusterProvec(key):
    clusterProvec = []
    for item in rep_fold[key]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def EC_clusterProvec(EC):
    clusterProvec = []
    for item in ec_id[EC]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def cluster_cos_sim(cluster):

    # Reshape the cluster if needed
    cluster = np.array(cluster).reshape(len(cluster), -1)

    # Compute pairwise cosine similarities
    similarities = cosine_similarity(cluster)
    # 去掉对角线的值
    np.fill_diagonal(similarities, 0)
    mask = np.triu(np.ones(similarities.shape), k=1).astype(bool)
    similarities_half = similarities[mask]
    return similarities_half

def cluster_cos_sim_full(cluster):

    # Reshape the cluster if needed
    cluster = np.array(cluster).reshape(len(cluster), -1)

    # Compute pairwise cosine similarities
    similarities = cosine_similarity(cluster)
    
    return similarities


In [ ]:
emb_dic = {}
with h5py.File(proVec_embfile , 'r') as hf:
        for key in hf.keys():
            emb_dic[key]=hf[key]



In [ ]:
hf = h5py.File(proVec_embfile , 'r') 

In [ ]:
len(hf.keys())

In [ ]:
foldseek_clusterProvec('A1JNT5')

In [ ]:
cluster_cos_sim_full(foldseek_clusterProvec('A1JNT5'))

In [ ]:
case_uniFun = cluster_cos_sim_full(foldseek_clusterProvec('Q7N2V3'))

In [ ]:
case_uniFun

In [ ]:
case_no_uniFun = cluster_cos_sim_full(foldseek_clusterProvec('P20921'))

In [ ]:
rep_fold['P20921']

In [ ]:
prosims = foldseek_clusterProvec('P20921')
prosims1 = prosims[-3:]
prosims2 = prosims[:-3]

In [ ]:
prosims1.extend(prosims2)

In [ ]:
rep_fold['Q7N2V3']

In [ ]:
outfile = open('../local_snapshot/data/afcase_uni_fun.txt','w')
for item in fun_fold['Q7N2V3']:
    outfile.write(item + '\n')

In [ ]:
outfile1 = open('../local_snapshot/data/afcase_uni_nofun.txt','w')
for item in fun_fold['P20921']:
    outfile1.write(item + '\n')

In [ ]:
for item in rep_fold['Q7N2V3']:
    if item in sel_struc:
        k = sel_keys[sel_struc.index(item)]
        print(rep_seq[k])
        print(id_ec[item])
    else:
        print(rep_seq[item])
        print(id_ec[item])

In [ ]:
['B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6',

'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8']


In [ ]:
import numpy as np

# 标签列表
labels = ['B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6', 'B5F0I6',

'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8', 'Q834L8']


# 创建一个空的相似性矩阵
num_members = len(labels)
similarity_matrix = np.zeros((num_members, num_members))

# 填充相似性矩阵
for i in range(num_members):
    for j in range(num_members):
        if labels[i] == labels[j]:
            similarity_matrix[i, j] = 1
        else:
            similarity_matrix[i, j] = 0.9

In [ ]:

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



# 计算余弦相似度
similarities = case_no_uniFun

# 使用seaborn绘制热图
sns.heatmap(similarities, cmap="YlGnBu", vmin=0.75, vmax=1)
#plt.title("Cosine Similarity Heatmap")
plt.xlabel("Sample of P20921 cluster")
plt.ylabel("Sample of P20921 cluster")
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



# 计算余弦相似度
similarities = case_no_uniFun

# 使用seaborn绘制热图
sns.heatmap(similarities, cmap="YlGnBu", vmin=0.75, vmax=1)
#plt.title("Cosine Similarity Heatmap")
plt.xlabel("Sample of P20921 cluster")
plt.ylabel("Sample of P20921 cluster")
plt.show()


In [ ]:
fun_fold_Prosim = {}
for key in fun_fold.keys():
    fun_fold_Prosim[key] = cluster_cos_sim(foldseek_clusterProvec(key))

In [ ]:
import pandas as pd
alignfile = pd.read_csv('../local_snapshot/data/alignment_db/afcase_uninofun', delimiter="\t", header=None)

In [ ]:
sns.kdeplot(alignfile[2].values)

In [ ]:
fun_repfold_Prosim = {}
for key in rep_fold.keys():
    fun_repfold_Prosim[key] = cluster_cos_sim(repfoldseek_clusterProvec(key))

In [ ]:
import h5py
proVec_embfileFS = '../local_snapshot/data/Embedding/RepFunFold_sep.h5'
proVec_embfileFN = '../local_snapshot/data/Embedding/RepFunFold_No.h5'
proVec_embfileF = '../local_snapshot/data/Embedding/RepFunFold.h5'
proVec_embfileES = '../local_snapshot/data/Embedding/RepFunEC_sep.h5'
proVec_embfileEN = '../local_snapshot/data/Embedding/RepFunEC_No.h5'
proVec_embfileE = '../local_snapshot/data/Embedding/RepFunEC.h5'
#'../local_snapshot/data/Embedding/sep3rnu5_1000e_s_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/sep3sssrn_1000e_s_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/sep3sr_s_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/sep3as_c_128_mlp_aug_dic.h5'
#'../local_snapshot/data/Embedding/split_50_prott_triplet_CLEAN.h5'
def load_proVec(lookup, proVec_embfile):
    
    with h5py.File(proVec_embfile , 'r') as hf:
            dataset_name = lookup
            dataset = hf[dataset_name]


            data = dataset[:]

    #torch_data = torch.tensor(data)
    return data #torch_data.unsqueeze(0)


def cos_sim(vector1,vector2):
    # Compute dot product
    dot_product = np.dot(vector1, vector2)

    # Compute magnitudes
    magnitude1 = np.linalg.norm(vector1)
    magnitude2 = np.linalg.norm(vector2)

    # Compute cosine similarity
    similarity = dot_product / (magnitude1 * magnitude2)
    return similarity


from sklearn.metrics.pairwise import cosine_similarity

def foldseek_clusterProvec(key):
    clusterProvec = []
    for item in fun_fold[key]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def repfoldseek_clusterProvec(key):
    clusterProvec = []
    for item in rep_fold[key]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def EC_clusterProvec(EC):
    clusterProvec = []
    for item in ec_id[EC]:
        clusterProvec.append(load_proVec(item))
    return clusterProvec

def cluster_cos_sim(cluster):

    # Reshape the cluster if needed
    cluster = np.array(cluster).reshape(len(cluster), -1)

    # Compute pairwise cosine similarities
    similarities = cosine_similarity(cluster)
    # 去掉对角线的值
    np.fill_diagonal(similarities, 0)
    mask = np.triu(np.ones(similarities.shape), k=1).astype(bool)
    similarities_half = similarities[mask]
    return similarities_half

def cluster_cos_sim_full(cluster):

    # Reshape the cluster if needed
    cluster = np.array(cluster).reshape(len(cluster), -1)

    # Compute pairwise cosine similarities
    similarities = cosine_similarity(cluster)
    
    return similarities


In [ ]:
fun_simfold_Prosim = {}
for key in uni_nofun_fold_keys: #fun_fold.keys():
    fun_simfold_Prosim[key] = cluster_cos_sim_full(foldseek_clusterProvec(key))

In [ ]:
Sep_fold = fun_simfold_Prosim

In [ ]:
Sep_ec = fun_simfold_Prosim

In [ ]:
No_fold = fun_simfold_Prosim

In [ ]:
No_ec = fun_simfold_Prosim

In [ ]:
Max_fold = fun_simfold_Prosim

In [ ]:
Max_ec = fun_simfold_Prosim

In [ ]:

keys_size = {}
for key in fun_fold.keys():
    keys_size[key] = len(fun_fold[key])


In [ ]:
segments = [(2, 4), (4, 10), (10, 50), (50, 100),(100, 300),(300, 600),(600, 900)]


In [ ]:
all_sim = {segment: [] for segment in segments}
True_funClu = {segment: [] for segment in segments}
False_funClu = {segment: [] for segment in segments}
for key in fun_fold_Prosim.keys():
    if key not in sing_funfold:
        for item in segments:
            if keys_size[key]>= item[0] and keys_size[key] < item[1]:
                all_sim[item].extend(fun_fold_Prosim[key].flatten())
                if key in uni_fun_clu_key_nsingle:
                    True_funClu[item].extend(fun_fold_Prosim[key].flatten())
                else:
                    False_funClu[item].extend(fun_fold_Prosim[key].flatten())

In [ ]:
len(all_sim)

In [ ]:
import numpy as np
def WCSS(similarity_matrix):
# 将相似度矩阵转换为距离矩阵
    distance_matrix = 1 - similarity_matrix

# 假设cluster_center是簇的中心（这里只是一个示例）
    cluster_center = np.mean(distance_matrix, axis=0)

# 计算每个成员到簇中心的距离的平方
    distances_to_center_squared = np.sum((distance_matrix - cluster_center) ** 2, axis=1)

# 计算簇内部平方和
    return np.sum(distances_to_center_squared)



In [ ]:
np.array([[1.0, 0.8, 0.6],
                               [0.8, 1.0, 0.7],
                               [0.6, 0.7, 1.0]])

In [ ]:
fun_repfold_Prosim

In [ ]:
all_sim_W_Av = []
True_funClu_W_Av = []
False_funClu_W_Av = []

for key in fun_simfold_Prosim.keys():
    if len(fun_fold[key])>10:
        all_sim_W_Av.append(np.mean(fun_simfold_Prosim[key]))
        if key in uni_fun_clu_key_nsingle_fold:
            True_funClu_W_Av.append(np.mean(fun_simfold_Prosim[key]))
        else:
            False_funClu_W_Av.append(np.mean(fun_simfold_Prosim[key]))


In [ ]:
all_sim_W_var = []
True_funClu_W_var = []
False_funClu_W_var = []

for key in fun_simfold_Prosim.keys():
    if len(fun_fold[key])>10:
        all_sim_W_var.append(np.var(fun_simfold_Prosim[key]))
        if key in uni_fun_clu_key_nsingle_fold:
            True_funClu_W_var.append(np.var(fun_simfold_Prosim[key]))
        else:
            False_funClu_W_var.append(np.var(fun_simfold_Prosim[key]))


In [ ]:
def cro_sim(key, ec, fold):
    clusterProvec = []
    for item in fun_fold[key]:
        clusterProvec.append(cos_sim(load_proVec(item,ec),load_proVec(item,fold)))
    return clusterProvec


In [ ]:
import numpy as np
from sklearn.metrics import mutual_info_score
def compute_mutual_information(var1, var2):
    # 计算 var1 和 var2 的联合分布
    joint_distribution = compute_joint_distribution(var1, var2)
    
    # 计算 var1 和 var2 的各自分布
    p_var1 = compute_marginal_distribution(var1)
    p_var2 = compute_marginal_distribution(var2)
    
    # 计算互信息
    mi = 0.0
    for (value1, value2), joint_prob in joint_distribution.items():
        if joint_prob > 0 and p_var1[value1] > 0 and p_var2[value2] > 0:
            mi += joint_prob * np.log(joint_prob / (p_var1[value1] * p_var2[value2]))
    
    return mi

def cro_sim(key, ec, fold):
    Var1 = []
    Var2 = []
    for item in fun_fold[key]:
        Var1.append(load_proVec(item, ec))
        Var2.append(load_proVec(item, fold))
    
    Mutual_Infor = mutual_info_score(np.array(Var1).flatten(), np.array(Var2).flatten())
    return Mutual_Infor


In [ ]:
all_sim_W_Av = []
True_funClu_W_Av = []
False_funClu_W_Av = []

for key in Max_ec.keys():
    #if len(fun_fold[key])>10:
        all_sim_W_Av.append(cro_sim(key, proVec_embfileE, proVec_embfileF))
        
        True_funClu_W_Av.append(cro_sim(key, proVec_embfileEN, proVec_embfileFN))
       
        False_funClu_W_Av.append(cro_sim(key, proVec_embfileES, proVec_embfileFS))

In [ ]:
key = 'P20921'
print(cro_sim(key, proVec_embfileE, proVec_embfileF),cro_sim(key, proVec_embfileEN, proVec_embfileFN),cro_sim(key, proVec_embfileES, proVec_embfileFS))

In [ ]:
all_sim_W_var = []
True_funClu_W_var = []
False_funClu_W_var = []

for key in fun_simfold_Prosim.keys():
   
        all_sim_W_var.append(np.var(cro_sim(key, proVec_embfileE, proVec_embfileF)))
       
        True_funClu_W_var.append(np.var(cro_sim(key, proVec_embfileEN, proVec_embfileFN)))
       
        False_funClu_W_var.append(np.var(cro_sim(key, proVec_embfileES, proVec_embfileFS)))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# 创建示例数据

# 将数据转换为DataFrame
df = pd.DataFrame({
    'Data1': all_sim_W_Av,
    'Data2': True_funClu_W_Av,
    'Data3': False_funClu_W_Av
})

# 将数据转换为长格式
df_long = pd.melt(df.reset_index(), id_vars=['index'], value_vars=['Data1', 'Data2', 'Data3'],
                  var_name='Category', value_name='Value')

# 绘制箱线图
plt.figure(figsize=(12, 6))
sns.boxplot(x='Category', y='Value', data=df_long, width=0.4)

# 添加每个数据点的比较连线
for i in range(len(df)):
    plt.plot(['Data1', 'Data2', 'Data3'], [df['Data1'][i], df['Data2'][i], df['Data3'][i]], color='gray', alpha=0.5)

# 添加标签和标题
plt.xlabel('Category')
plt.ylabel('Values')
plt.title('Boxplots with Comparison Lines for Each Data Point')

# 显示图像
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = all_sim_W_Av
data2 = True_funClu_W_Av
data3 = False_funClu_W_Av
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Vertical Violin Plot - Data Distribution')
#plt.xlabel('Density')
plt.ylabel('Mean')
#plt.ylabel('Various')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Minimum Entropy', 'Null','Maximum Entropy'])
plt.xticks(rotation=15)
#plt.ylim(0.0, 0.25) 
#plt.ylim(0, 0.00000002) 
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = all_sim_W_var
data2 = True_funClu_W_var
data3 = False_funClu_W_var
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Vertical Violin Plot - Data Distribution')
#plt.xlabel('Density')
plt.ylabel('Various')
#plt.ylabel('Various')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Minimum Entropy', 'Null','Maximum Entropy'])
plt.xticks(rotation=15)
#plt.ylim(0.9995, 1.00005) 
plt.ylim(-0.045, 0.345) 
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = all_sim_pro_A
data2 = True_funClu_pro_A
data3 = False_funClu_pro_A
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Vertical Violin Plot - Data Distribution')
#plt.xlabel('Density')
plt.ylabel('within_cluster_sum_of_squares')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['All', 'EC consistent','EC not consistent'])
plt.xticks(rotation=15)
plt.ylim(0, 1000) 
plt.show()

In [ ]:
all_sim_A = []
True_funClu_A = []
False_funClu_A = []
for key in fun_repfold_Prosim.keys():
    if key not in sing_fold:
        all_sim_A.extend(fun_repfold_Prosim[key].flatten())
        if key in uni_fun_clu_key_nsingle_fold:
            True_funClu_A.extend(fun_repfold_Prosim[key].flatten())
        else:
            False_funClu_A.extend(fun_repfold_Prosim[key].flatten())

In [ ]:
all_sim_d_A = []
True_funClu_d_A = []
False_funClu_d_A = []

for key in fun_repfold_Prosim.keys():
    if key not in sing_fold:
        all_sim_d_A.append((max(fun_repfold_Prosim[key].flatten())-min(fun_repfold_Prosim[key].flatten())))
        if key in uni_fun_clu_key_nsingle_fold:
            True_funClu_d_A.append((max(fun_repfold_Prosim[key].flatten())-min(fun_repfold_Prosim[key].flatten())))
        else:
            False_funClu_d_A.append((max(fun_repfold_Prosim[key].flatten())-min(fun_repfold_Prosim[key].flatten())))

               

In [ ]:
all_sim_d = {segment: [] for segment in segments}
True_funClu_d = {segment: [] for segment in segments}
False_funClu_d = {segment: [] for segment in segments}

for key in fun_fold_Prosim.keys():
    if key not in sing_funfold:
        for item in segments:
            if keys_size[key]>= item[0] and keys_size[key] < item[1]:
                all_sim_d[item].append((max(fun_fold_Prosim[key].flatten())-min(fun_fold_Prosim[key].flatten())))
                if key in uni_fun_clu_key_nsingle:
                    True_funClu_d[item].append((max(fun_fold_Prosim[key].flatten())-min(fun_fold_Prosim[key].flatten())))
                else:
                    False_funClu_d[item].append((max(fun_fold_Prosim[key].flatten())-min(fun_fold_Prosim[key].flatten())))

In [ ]:
max(fun_fold_Prosim['A1JNT5'].flatten())-min(fun_fold_Prosim['A1JNT5'].flatten())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = all_simEC_d_A
data2 = True_foldClu_d_A
data3 = False_foldClu_d_A
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Vertical Violin Plot - Data Distribution')
#plt.xlabel('Density')
plt.ylabel('cosine similarity difference')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['All pairs', 'Pairs within consistent fold clusters','Pairs without consistent fold clusters'])
plt.xticks(rotation=15)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 两个数据列表
data1 = all_simEC_A
data2 = True_foldClu_A
data3 = False_foldClu_A
# 使用 Seaborn 绘制两个响铃图在同一个图上
ax = sns.violinplot(data=[data1, data2,data3], palette="muted", orient='v')

#plt.title('Vertical Violin Plot - Data Distribution')
#plt.xlabel('Density')
plt.ylabel('cosine similarity')
plt.grid(False)
# 设置 x 轴的刻度和标签
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['All pairs', 'Pairs within consistant fold clusters','Pairs without consistant fold clusters'])
plt.xticks(rotation=15)
plt.show()

In [ ]:
sns.kdeplot(all_sim)

In [ ]:
fun_prosim = {}
for key in fun_clu.keys():
    fun_prosim[key] = cluster_cos_sim(EC_clusterProvec(key))

In [ ]:
uni_fold_clu_key_nsingle

In [ ]:
all_simEC_A = []
True_foldClu_A = []
False_foldClu_A = []
for key in fun_prosim.keys():
    if key not in sing_fold_fun:
        all_simEC_A.extend(fun_prosim[key].flatten())
        if key in uni_fold_clu_key_nsingle:
            True_foldClu_A.extend(fun_prosim[key].flatten())
        else:
            False_foldClu_A.extend(fun_prosim[key].flatten())

            
all_simEC_d_A = []
True_foldClu_d_A = []
False_foldClu_d_A = []

for key in fun_prosim.keys():
    if key not in sing_fold_fun:
        all_simEC_d_A.append((max(fun_prosim[key].flatten())-min(fun_prosim[key].flatten())))
        if key in uni_fold_clu_key_nsingle:
            True_foldClu_d_A.append((max(fun_prosim[key].flatten())-min(fun_prosim[key].flatten())))
        else:
            False_foldClu_d_A.append((max(fun_prosim[key].flatten())-min(fun_prosim[key].flatten())))

 

In [ ]:
Fun_keys = open('../local_snapshot/data/foldseek_Clu_key.txt','w')
for key in fun_fold.keys():
    Fun_keys.write(key+'\n')
Fun_keys.close()

In [ ]:
file_pre = '../local_snapshot/data/all/new_foldseek_key_cluster.tsv'

In [ ]:
rep_pre = {}

seqfile = open(file_pre,'r')
for line in seqfile:
    key = line.split('\t')[0].split('.')[0]
    if key in rep_pre.keys():
        rep_pre[key].append(line.split('\t')[1].split('.')[0])
    else:
        rep_pre[key] = []
        rep_pre[key].append(line.split('\t')[1].split('.')[0])
        

In [ ]:
len(rep_pre.keys())

In [ ]:
keys = []
for key in fun_fold.keys():
    keys.append(key)

In [ ]:
keys_pre = []
items_pre = []
for key in rep_pre.keys():
    for item in rep_pre[key]:
        keys_pre.append(key)
        items_pre.append(item)

In [ ]:
fun_check_pre = {}
for item in items_pre:
    if item in keys:
         fun_check_pre[keys_pre[items_pre.index(item)]]=id_ec[item]
        

In [ ]:
len(fun_check_pre)

In [ ]:
len(list(set(keys).intersection(set(items_pre))))

In [ ]:
len(keys_pre)

In [ ]:
test_file = open('../local_snapshot/data/new.txt','r')
test_data = []
for line in test_file:
    test_data.append(line.split('\n')[0])
    

In [ ]:
len(test_data)

In [ ]:
test_pre = {}
y_pre = []
test_true = {}
y_true = []
for item in test_data:
    if item in items_pre:
        if keys_pre[items_pre.index(item)] in fun_check_pre.keys():
            test_pre[item] = fun_check_pre[keys_pre[items_pre.index(item)]]
            y_pre.append(fun_check_pre[keys_pre[items_pre.index(item)]])
            test_true[item] = id_ec_test[item]
            y_true.append(id_ec_test[item])

In [ ]:
len(test_pre)

In [ ]:
id_ec_test, ec_id_test =  get_ec_id_dict(dirs+'data/new' + '.csv')

In [ ]:
len(test_true)

In [ ]:
Pos_pre = []
for key in test_pre.keys():
    if test_pre[key] == test_true[key]:
        Pos_pre.append(key)

In [ ]:
len(Pos_pre)

In [ ]:
fold_universe_file = '../local_snapshot/data/1-AFDBClusters-entryId_repId_taxId.tsv'
fold_universe = open(fold_universe_file,'r')
fold_universe_data = {}
for line in fold_universe:
    key = line.split('\t')[1]
    if key in fold_universe_data.keys():
        fold_universe_data[key].append(line.split('\t')[0])
    else:
        fold_universe_data[key] = []
        fold_universe_data[key].append(line.split('\t')[0])

In [ ]:
len(fold_universe_data)

In [ ]:
fold_universe_keys = []
items_key = []
items_value = []
for key in fold_universe_data.keys():
    fold_universe_keys.append(key)
    for item in fold_universe_data[key]:
        items_key.append(key)
        items_value.append(item)
    
    
fun_fold_keys = []
fun_items_key = []
fun_items_value = []
for key in fun_fold.keys():
    fun_fold_keys.append(key)
    for item in fun_fold[key]:
        fun_items_key.append(key)
        fun_items_value.append(item)
    

In [ ]:
fun_universe_key = []
fun_local_key = []
key_items = []
for item in fun_items_value:
    if item in items_value:
        key_items.append(item)
        fun_universe_key.append(items_key[items_value.index(item)])
        fun_local_key.append(fun_items_key[fun_items_value.index(item)])

In [ ]:
key_items = []
for item in fun_fold.keys():
    key_items.append(item)

In [ ]:
len(set(fun_universe_key))

In [ ]:
len(set(fun_local_key))

In [ ]:
len(list(set(items_value).intersection(set(key_items))))

In [ ]:
len(fun_items_value)

In [ ]:
len(items_value)

In [ ]:
Allfold_universe_file = '../local_snapshot/data/5-allmembers-repId-entryId-cluFlag-taxId.tsv'
Allfold_universe = open(Allfold_universe_file,'r')
Allfold_universe_data = {}
for line in Allfold_universe:
    key = line.split('\t')[0]
    if key in Allfold_universe_data.keys():
        Allfold_universe_data[key].append(line.split('\t')[1])
    else:
        Allfold_universe_data[key] = []
        Allfold_universe_data[key].append(line.split('\t')[1])

In [ ]:
len(Allfold_universe_data)

In [ ]:
Allfold_universe_keys = []
Allitems_key = []
Allitems_value = []
for key in Allfold_universe_data.keys():
    Allfold_universe_keys.append(key)
    for item in Allfold_universe_data[key]:
        Allitems_key.append(key)
        Allitems_value.append(item)
    

In [ ]:
universe_fun_fold_key = {}
for key in fun_fold.keys():
    universe_fun_fold_key[key]=Allitems_key[Allitems_value.index(key)]
    